In [ ]:
import os
from PIL import Image

input_base_dir = '/home/yale/Desktop/spectrograms/'
output_base_dir = '/home/yale/Desktop/spectrograms_final'

# Cropping parameters
left_crop = 94
bottom_crop = 47
top_crop_fixed = 5

os.makedirs(output_base_dir, exist_ok=True)

# Go over each folder directly under spectrograms/
folders = sorted([f for f in os.listdir(input_base_dir) if os.path.isdir(os.path.join(input_base_dir, f))])

for folder in folders:
    input_folder_path = os.path.join(input_base_dir, folder)
    output_folder_path = os.path.join(output_base_dir, folder)
    os.makedirs(output_folder_path, exist_ok=True)

    # Only process .png files (skip CSVs)
    png_files = [f for f in os.listdir(input_folder_path) if f.lower().endswith('.png')]

    for image_file in png_files:
        image_path = os.path.join(input_folder_path, image_file)
        output_image_path = os.path.join(output_folder_path, image_file)

        try:
            original_img = Image.open(image_path)

            # Step 1: Crop axes
            axis_cropped_img = original_img.crop((
                left_crop,
                top_crop_fixed,
                original_img.width,
                original_img.height - bottom_crop
            ))

            # Step 2: Square crop
            width, height = axis_cropped_img.size
            if height > width:
                top_crop = height - width
                final_img = axis_cropped_img.crop((0, top_crop, width, height))
            elif width > height:
                extra = width - height
                left = extra // 2
                right = width - (extra - left)
                final_img = axis_cropped_img.crop((left, 0, right, height))
            else:
                final_img = axis_cropped_img  # already square

            # Step 3: Save
            final_img.save(output_image_path)

        except Exception as e:
            print(f"Failed to process {image_path}: {e}")

print("All PNG images processed and saved! CSVs ignored.")
